# LCZ Species Analysis

Analyses the spatial distribution of Random Forest allergenic tree species
predictions across Local Climate Zone (LCZ) classes.

**Two analyses are performed:**
1. **Public trees (test set)** — trees with ground-truth species labels from
   the municipal inventory. Classification metrics (precision, recall, overall
   accuracy) are computed per LCZ class.
2. **Private trees** — trees without ground-truth labels. Only prediction counts
   per species per LCZ class are reported, since no reference labels exist.

The LCZ raster is obtained from:
   https://zenodo.org/records/8419340

**Import libraries**

In [ ]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import EXTERNAL_DIR, PROCESSED_DIR

**Define paths**

In [ ]:
# Random Forest multi-class predictions from feature confifugration 2 (all trees — public and private)
RF_PREDICTIONS_DIR = PROCESSED_DIR / "rf_multiclass/enschede_rf_multiclass_predictions_fc2_v01.shp"

# LCZ raster for Enschede
LCZ_RASTER = EXTERNAL_DIR / "lcz_raster/lcz_v3.tif"

**Define LCZ class labels**

In [ ]:
LCZ_LABELS = {
    1:  "Compact high-rise",   2: "Compact mid-rise",   3: "Compact low-rise",
    4:  "Open high-rise",      5: "Open mid-rise",       6: "Open low-rise",
    7:  "Lightweight low-rise",8: "Large low-rise",      9: "Sparsely built",
    10: "Heavy industry",      11: "Dense trees",        12: "Scattered trees",
    13: "Bush, scrub",         14: "Low plants",         15: "Bare rock or paved",
    16: "Bare soil or sand",   17: "Water",
}

BUILT_UP_NAMES   = [LCZ_LABELS[i] for i in range(1, 11)]
LAND_COVER_NAMES = [LCZ_LABELS[i] for i in range(11, 18)]

CLASS_NAMES  = ["Betula", "Alnus", "Fraxinus", "Quercus", "Platanus"]
SPECIES_MAP  = {0: "Betula", 1: "Alnus", 2: "Fraxinus", 3: "Quercus", 4: "Platanus"}

**Load predictions and sample LCZ raster at crown centroids**

In [ ]:
trees = gpd.read_file(RF_PREDICTIONS_DIR)

# Sample the LCZ raster value at each crown centroid
with rasterio.open(LCZ_RASTER) as src:
    trees = trees.to_crs(src.crs)
    centroids = trees.geometry.centroid
    coords    = [(geom.x, geom.y) for geom in centroids]
    lcz_vals  = list(src.sample(coords))

trees["LCZ"]      = [val[0] for val in lcz_vals]
trees["LCZ_name"] = trees["LCZ"].map(LCZ_LABELS)

# Map predicted integer class to species name
trees["pred_species_name"] = trees["pred_speci"].map(SPECIES_MAP)

**Split into public test trees and private trees**

In [ ]:
# Public trees: allergenic label from municipal inventory (is_allerge == 1)
# These have ground-truth species labels and are used to compute metrics.
public_test_trees = trees[trees["is_allerge"] == 1].copy()

# Private trees: no ground-truth label (is_allerge is NaN)
# Only prediction counts are reported for these.
private_trees = trees[trees["is_allerge"].isna()].copy()

print(f"Public test trees:  {len(public_test_trees)}")
print(f"Private trees:      {len(private_trees)}")

**Function: styled metric table (public trees)**

In [ ]:
def generate_styled_metric_table(df, name_list, class_names):
    """
    Compute per-LCZ precision, recall, and overall accuracy for public test trees
    and return a styled DataFrame with colour-gradient highlighting.

    Args:
        df:         GeoDataFrame of public test trees with 'LCZ_name',
                    'species_na' (true label), and 'pred_species_name' columns.
        name_list:  List of LCZ class names to include (built-up or land-cover).
        class_names: Ordered list of species names matching model class IDs.

    Returns:
        pandas Styler object ready for display in a Jupyter notebook.
    """
    results = []

    for lcz_name in name_list:
        lcz_data = df[df["LCZ_name"] == lcz_name]
        if len(lcz_data) < 1:
            continue

        y_true = lcz_data["species_na"]
        y_pred = lcz_data["pred_species_name"]

        acc = accuracy_score(y_true, y_pred)
        precision, recall, _, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=class_names, average=None, zero_division=0
        )

        row = {"LCZ Type": lcz_name}
        for i, species in enumerate(class_names):
            row[f"Count_{species}"]     = (y_pred == species).sum()
            row[f"Precision_{species}"] = precision[i]
            row[f"Recall_{species}"]    = recall[i]
        row["Overall_Accuracy"] = acc
        results.append(row)

    metric_df = pd.DataFrame(results).set_index("LCZ Type")

    count_cols = [f"Count_{s}"     for s in class_names]
    prec_cols  = [f"Precision_{s}" for s in class_names]
    rec_cols   = [f"Recall_{s}"    for s in class_names]
    metric_df  = metric_df[count_cols + prec_cols + rec_cols + ["Overall_Accuracy"]]

    top    = (["Count"] * len(class_names) + ["Precision"] * len(class_names)
              + ["Recall"] * len(class_names) + ["Overall Accuracy"])
    bottom = class_names + class_names + class_names + [""]
    metric_df.columns = pd.MultiIndex.from_arrays([top, bottom])

    styled = (
        metric_df.style
        .background_gradient(cmap="Reds",
                             subset=pd.IndexSlice[:, pd.IndexSlice["Count", :]],
                             vmin=0, vmax=metric_df["Count"].max().max())
        .background_gradient(cmap="Reds",
                             subset=pd.IndexSlice[:, pd.IndexSlice["Precision", :]],
                             vmin=0, vmax=1)
        .background_gradient(cmap="Reds",
                             subset=pd.IndexSlice[:, pd.IndexSlice["Recall", :]],
                             vmin=0, vmax=1)
        .background_gradient(cmap="Reds",
                             subset=pd.IndexSlice[:, pd.IndexSlice["Overall Accuracy", ""]],
                             vmin=0, vmax=1)
        .format("{:.0f}", subset=pd.IndexSlice[:, ("Count", slice(None))])
        .format("{:.2f}", subset=pd.IndexSlice[:, ("Precision", slice(None))])
        .format("{:.2f}", subset=pd.IndexSlice[:, ("Recall", slice(None))])
        .format("{:.2f}", subset=pd.IndexSlice[:, ("Overall Accuracy", "")])
    )
    return styled

**Function: prediction count table (private trees)**

In [ ]:
def generate_count_table(df, name_list, class_names):
    """
    Count private tree predictions per species per LCZ class and return a
    styled DataFrame.

    Args:
        df:         GeoDataFrame of private trees with 'LCZ_name' and
                    'pred_species_name' columns.
        name_list:  List of LCZ class names to include.
        class_names: Ordered list of species names.

    Returns:
        pandas Styler object.
    """
    sub_df      = df[df["LCZ_name"].isin(name_list)]
    count_table = (sub_df.groupby("LCZ_name")["pred_species_name"]
                         .value_counts()
                         .unstack(fill_value=0))

    for species in class_names:
        if species not in count_table.columns:
            count_table[species] = 0

    count_table = count_table[class_names]
    count_table.columns.name = None

    styled = (
        count_table.style
        .background_gradient(cmap="Reds",
                             vmin=count_table.min().min(),
                             vmax=count_table.max().max())
    )
    return styled

**Function: confusion matrix per LCZ group**

In [ ]:
def plot_confusion_matrix_group(df, lcz_group, class_names, title):
    """
    Plot a confusion matrix for public test trees within a given set of LCZ classes.

    Args:
        df:          GeoDataFrame of public test trees.
        lcz_group:   List of LCZ class names to include.
        class_names: Ordered list of species names.
        title:       Figure title.
    """
    subset = df[df["LCZ_name"].isin(lcz_group)]

    cm = confusion_matrix(
        subset["pred_species_name"],
        subset["species_na"],
        labels=class_names,
    )

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.tight_layout()
    plt.show()

**Run analysis**

In [ ]:
# Public trees — metric tables per LCZ group
display(generate_styled_metric_table(public_test_trees, BUILT_UP_NAMES,   CLASS_NAMES))
display(generate_styled_metric_table(public_test_trees, LAND_COVER_NAMES, CLASS_NAMES))

# Private trees — count tables per LCZ group
display(generate_count_table(private_trees, BUILT_UP_NAMES,   CLASS_NAMES))
display(generate_count_table(private_trees, LAND_COVER_NAMES, CLASS_NAMES))

# Confusion matrices for public trees per LCZ group
plot_confusion_matrix_group(public_test_trees, BUILT_UP_NAMES,
                            CLASS_NAMES, "Confusion Matrix - Built-up LCZ")
plot_confusion_matrix_group(public_test_trees, LAND_COVER_NAMES,
                            CLASS_NAMES, "Confusion Matrix - Land-cover LCZ")